In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_curve, auc
from sklearn.preprocessing import label_binarize
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from tqdm import tqdm

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = 'convnext'

# Paths
OUTPUT_PATH = f'1_Feature_Extraction/{model_name}'
RESULTS_PATH = os.path.join(f'2_Model_Comparison/{model_name}')
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load feature data
train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f'train_features_{model_name}.csv'))
test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f'test_features_{model_name}.csv'))

# Extract features and labels
X_train = train_df[[f'feat_{i}' for i in range(1024)]].values
y_train = train_df['label'].map({'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}).values
X_test = test_df[[f'feat_{i}' for i in range(1024)]].values
y_test = test_df['label'].map({'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}).values

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
N_CLASSES = len(CLASSES)

# Plot settings
rcParams['font.family'] = 'Times New Roman'
rcParams['axes.titlesize'] = 28
rcParams['axes.titlepad'] = 20
rcParams['axes.labelsize'] = 23
rcParams['xtick.labelsize'] = 18
rcParams['ytick.labelsize'] = 18
rcParams['legend.fontsize'] = 16
rcParams['lines.linewidth'] = 3
rcParams['axes.linewidth'] = 2

# Custom Dataset for PyTorch models
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Attention Layer
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

# AttCNN Model
class AttCNN(nn.Module):
    def __init__(self, input_dim=1024, num_classes=4):
        super(AttCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.attention = Attention(128 * (input_dim // 4))
        self.fc = nn.Linear(128 * (input_dim // 4), num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.attention(x.unsqueeze(1))
        x = self.fc(x)
        return x

# AttGRU Model
class AttGRU(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=512, num_classes=4):
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

# Training function for PyTorch models
def train_model(model, train_loader, test_loader, num_epochs=50, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_losses.append(running_loss / len(train_loader))
        
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
        test_losses.append(test_loss / len(test_loader))
    
    return train_losses, test_losses

# Evaluation function
def evaluate_model(model, X_train, y_train, X_test, y_test, model_type='sklearn'):
    if model_type == 'sklearn':
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        y_prob_test = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    else:  # PyTorch
        model.eval()
        with torch.no_grad():
            X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
            y_pred_train = model(X_train_tensor).argmax(dim=1).cpu().numpy()
            y_pred_test = model(X_test_tensor).argmax(dim=1).cpu().numpy()
            y_prob_test = torch.softmax(model(X_test_tensor), dim=1).cpu().numpy()
    
    # Metrics
    metrics = {}
    metrics['ACC'] = accuracy_score(y_test, y_pred_test)
    metrics['AUC'] = roc_auc_score(y_test, y_prob_test, multi_class='ovr') if y_prob_test is not None else 0
    metrics['PRE'] = precision_score(y_test, y_pred_test, average='macro')
    metrics['SN'] = recall_score(y_test, y_pred_test, average='macro')  # Sensitivity
    metrics['SP'] = recall_score(y_test, y_pred_test, average='macro')  # Specificity (simplified)
    metrics['F1'] = f1_score(y_test, y_pred_test, average='macro')
    metrics['MCC'] = matthews_corrcoef(y_test, y_pred_test)
    
    return metrics, y_prob_test, y_pred_test

# Plot training/loss curves
def plot_curves(train_losses, test_losses, model_name):
    plt.figure(figsize=(8, 8))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f'{model_name} Training and Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_loss_curve.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_loss_curve.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()

# Plot ROC curves
def plot_roc(y_test, y_prob, model_name):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    
    plt.figure(figsize=(8, 8))
    for i in range(N_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f'{CLASSES[i]} (AUC = {roc_auc[i]:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'{model_name} ROC Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_roc_curve.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_roc_curve.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()

# Main execution
models = {
    'SVM': SVC(probability=True, kernel='rbf'),
    'CatBoost': CatBoostClassifier(iterations=100, verbose=0),
    'LightGBM': LGBMClassifier(n_estimators=100),
    'AttCNN': AttCNN().to(device),
    'AttGRU': AttGRU().to(device)
}

results = []
train_dataset = FeatureDataset(X_train, y_train)
test_dataset = FeatureDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name in ['SVM', 'CatBoost', 'LightGBM']:
        model.fit(X_train, y_train)
        metrics, y_prob, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, 'sklearn')
    else:  # PyTorch models
        train_losses, test_losses = train_model(model, train_loader, test_loader)
        metrics, y_prob, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, 'pytorch')
        plot_curves(train_losses, test_losses, model_name)
    
    # Plot ROC
    if y_prob is not None:
        plot_roc(y_test, y_prob, model_name)
    
    # Store results
    results.append({
        'Model': model_name,
        'ACC': metrics['ACC'],
        'AUC': metrics['AUC'],
        'PRE': metrics['PRE'],
        'SP': metrics['SP'],
        'SN': metrics['SN'],
        'F1': metrics['F1'],
        'MCC': metrics['MCC']
    })
    print(f"{model_name} Metrics: {metrics}")

# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS_PATH, f'model_performance_{model_name}.csv'), index=False)
print(f"Saved model performance to {os.path.join(RESULTS_PATH, f'model_performance_{model_name}.csv')}")

print("Evaluation complete!")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_curve, auc, confusion_matrix
from sklearn.preprocessing import label_binarize
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from tqdm import tqdm

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = 'densenet121'

# Paths
OUTPUT_PATH = f'1_Feature_Extraction/{model_name}'
RESULTS_PATH = os.path.join(f'2_Model_Comparison/{model_name}')
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load feature data
train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f'train_features_{model_name}.csv'))
test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f'test_features_{model_name}.csv'))

# Extract features and labels
X_train = train_df[[f'feat_{i}' for i in range(1024)]].values
y_train = train_df['label'].map({'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}).values
X_test = test_df[[f'feat_{i}' for i in range(1024)]].values
y_test = test_df['label'].map({'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}).values

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
N_CLASSES = len(CLASSES)

# Plot settings
rcParams['font.family'] = 'Times New Roman'
rcParams['axes.titlesize'] = 28
rcParams['axes.titlepad'] = 20
rcParams['axes.labelsize'] = 23
rcParams['xtick.labelsize'] = 18
rcParams['ytick.labelsize'] = 18
rcParams['legend.fontsize'] = 16
rcParams['lines.linewidth'] = 3
rcParams['axes.linewidth'] = 2

# Custom Dataset for PyTorch models
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Attention Layer
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

# Optimized AttCNN Model (reduced complexity)
class AttCNN(nn.Module):
    def __init__(self, input_dim=1024, num_classes=4):
        super(AttCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.attention = Attention(64 * (input_dim // 4))
        self.fc = nn.Linear(64 * (input_dim // 4), num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.attention(x.unsqueeze(1))
        x = self.fc(x)
        return x

# AttGRU Model
class AttGRU(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=512, num_classes=4):
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

# Training function with progress bar and checkpointing
def train_model(model, train_loader, test_loader, num_epochs=50, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    
    checkpoint_path = os.path.join(RESULTS_PATH, f'{model.__class__.__name__}_checkpoint.pth')
    
    # Progress bar for epochs
    for epoch in tqdm(range(num_epochs), desc=f"Training {model.__class__.__name__}"):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_losses.append(running_loss / len(train_loader))
        
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
        test_losses.append(test_loss / len(test_loader))
    
    # Save model checkpoint
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Saved {model.__class__.__name__} checkpoint to {checkpoint_path}")
    
    return train_losses, test_losses

# Evaluation function
def evaluate_model(model, X_train, y_train, X_test, y_test, model_type='sklearn'):
    if model_type == 'sklearn':
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        y_prob_test = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    else:  # PyTorch
        model.eval()
        with torch.no_grad():
            X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
            y_pred_train = model(X_train_tensor).argmax(dim=1).cpu().numpy()
            y_pred_test = model(X_test_tensor).argmax(dim=1).cpu().numpy()
            y_prob_test = torch.softmax(model(X_test_tensor), dim=1).cpu().numpy()
    
    # Metrics
    metrics = {}
    metrics['ACC'] = accuracy_score(y_test, y_pred_test)
    metrics['AUC'] = roc_auc_score(y_test, y_prob_test, multi_class='ovr') if y_prob_test is not None else 0
    metrics['PRE'] = precision_score(y_test, y_pred_test, average='macro')
    metrics['SN'] = recall_score(y_test, y_pred_test, average='macro')  # Sensitivity
    metrics['SP'] = recall_score(y_test, y_pred_test, average='macro')  # Specificity (simplified)
    metrics['F1'] = f1_score(y_test, y_pred_test, average='macro')
    metrics['MCC'] = matthews_corrcoef(y_test, y_pred_test)
    
    return metrics, y_prob_test, y_pred_test

# Plot training/loss curves
def plot_curves(train_losses, test_losses, model_name):
    plt.figure(figsize=(8, 8))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f'{model_name} Training and Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_loss_curve.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_loss_curve.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()

# Plot ROC curves
def plot_roc(y_test, y_prob, model_name):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    
    plt.figure(figsize=(8, 8))
    for i in range(N_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f'{CLASSES[i]} (AUC = {roc_auc[i]:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'{model_name} ROC Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_roc_curve.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_roc_curve.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()

# Plot confusion matrix
def plot_confusion_matrix(y_test, y_pred, model_name):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES,
                annot_kws={"size": 18})
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_confusion_matrix.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'{model_name}_confusion_matrix.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()

# Main execution
models = {
    'SVM': SVC(probability=True, kernel='rbf'),
    'CatBoost': CatBoostClassifier(iterations=100, verbose=0),
    'LightGBM': LGBMClassifier(n_estimators=100),
    'AttCNN': AttCNN().to(device),
    'AttGRU': AttGRU().to(device)
}

results = []
train_dataset = FeatureDataset(X_train, y_train)
test_dataset = FeatureDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name in ['SVM', 'CatBoost', 'LightGBM']:
        model.fit(X_train, y_train)
        metrics, y_prob, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, 'sklearn')
    else:  # PyTorch models
        train_losses, test_losses = train_model(model, train_loader, test_loader)
        metrics, y_prob, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, 'pytorch')
        plot_curves(train_losses, test_losses, model_name)
    
    # Plot ROC and Confusion Matrix
    if y_prob is not None:
        plot_roc(y_test, y_prob, model_name)
    plot_confusion_matrix(y_test, y_pred, model_name)
    
    # Store results
    results.append({
        'Model': model_name,
        'ACC': metrics['ACC'],
        'AUC': metrics['AUC'],
        'PRE': metrics['PRE'],
        'SP': metrics['SP'],
        'SN': metrics['SN'],
        'F1': metrics['F1'],
        'MCC': metrics['MCC']
    })
    print(f"{model_name} Metrics: {metrics}")

# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS_PATH, f'model_performance_{model_name}.csv'), index=False)
print(f"Saved model performance to {os.path.join(RESULTS_PATH, f'model_performance_{model_name}.csv')}")

print("Evaluation complete!")